# Gradient Descent

**Checkpoint:** CP0.2, Notebook 02  
**Goal:** implement gradient descent from scratch for one- and two-variable functions and explain how the learning rate changes convergence.

The update rule is `x_next = x - learning_rate * gradient(x)`. Gradient descent moves opposite the gradient because that is locally the direction of steepest decrease. Run the notebook from top to bottom.

## 1. One-variable function

Start with `f(x) = x^2 + 3x + 2`. Its derivative is `2x + 3`; setting it to zero gives the minimum at `x = -1.5`. The second derivative is `2`, so the gradient changes at a predictable rate. For this quadratic, a learning rate above `1` is too large and causes the iterates to diverge from the minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def f(x):
    return x**2 + 3 * x + 2


def gradient_f(x):
    return 2 * x + 3


def gradient_descent_1d(function, gradient, initial_x, learning_rate, max_steps=500, tolerance=1e-8):
    """Return the sequence of parameters and objective values."""
    x = float(initial_x)
    parameters = [x]
    values = [function(x)]
    converged = False

    for _ in range(max_steps):
        grad = gradient(x)
        if abs(grad) < tolerance:
            converged = True
            break

        x = x - learning_rate * grad
        parameters.append(x)
        values.append(function(x))

        if not np.isfinite(x) or not np.isfinite(values[-1]):
            break

    if abs(gradient(x)) < tolerance:
        converged = True

    return np.asarray(parameters), np.asarray(values), converged


minimum_x = -1.5
learning_rates_1d = {"small": 0.05, "moderate": 0.4, "too large": 1.1}
results_1d = {}

for label, learning_rate in learning_rates_1d.items():
    parameters, values, converged = gradient_descent_1d(
        f, gradient_f, initial_x=4.0, learning_rate=learning_rate
    )
    results_1d[label] = (parameters, values, converged)
    print(
        f"{label:>9} (lr={learning_rate}): steps={len(parameters) - 1:3d}, "
        f"x={parameters[-1]: .6g}, f(x)={values[-1]: .6g}, converged={converged}"
    )

assert results_1d["moderate"][2]
assert abs(results_1d["moderate"][0][-1] - minimum_x) < 1e-7
assert not results_1d["too large"][2]

In [ ]:
plt.figure(figsize=(8, 5))
for label, (_, values, _) in results_1d.items():
    plt.plot(range(len(values)), values, label=label)
plt.axhline(f(minimum_x), color="black", linestyle="--", linewidth=1, label="minimum")
plt.xlabel("Iteration")
plt.ylabel("f(x)")
plt.title("One-variable gradient descent")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
for label, (parameters, _, _) in results_1d.items():
    plt.plot(range(len(parameters)), parameters, label=label)
plt.axhline(minimum_x, color="black", linestyle="--", linewidth=1, label="minimum x")
plt.xlabel("Iteration")
plt.ylabel("x")
plt.title("Parameter updates in one dimension")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

A small learning rate makes cautious progress and can need many iterations. A moderate rate reaches the minimum faster here. With `lr = 1.1`, the update overshoots; the distance to the minimum grows rather than shrinking. This behavior follows from the update itself: the error is multiplied by `1 - 2 * lr` each iteration, whose magnitude is greater than one when `lr > 1`.

## 2. Two-variable function

Use `g(x, y) = (x - 2)^2 + 2(y + 1)^2`. Its minimum is `(2, -1)` and its gradient is `[2(x - 2), 4(y + 1)]`. The steeper curvature in the `y` direction makes the learning-rate stability limit smaller than in the first example: rates above `0.5` diverge for this quadratic.

In [ ]:
def g(point):
    x, y = point
    return (x - 2) ** 2 + 2 * (y + 1) ** 2


def gradient_g(point):
    x, y = point
    return np.array([2 * (x - 2), 4 * (y + 1)], dtype=float)


def gradient_descent_2d(function, gradient, initial_point, learning_rate, max_steps=1000, tolerance=1e-8):
    """Return parameter and objective histories for a two-variable function."""
    point = np.asarray(initial_point, dtype=float)
    parameters = [point.copy()]
    values = [function(point)]
    converged = False

    for _ in range(max_steps):
        grad = gradient(point)
        if np.linalg.norm(grad) < tolerance:
            converged = True
            break

        point = point - learning_rate * grad
        parameters.append(point.copy())
        values.append(function(point))

        if not np.all(np.isfinite(point)) or not np.isfinite(values[-1]):
            break

    if np.linalg.norm(gradient(point)) < tolerance:
        converged = True

    return np.asarray(parameters), np.asarray(values), converged


minimum_point = np.array([2.0, -1.0])
learning_rates_2d = {"small": 0.05, "moderate": 0.2, "too large": 0.55}
results_2d = {}

for label, learning_rate in learning_rates_2d.items():
    parameters, values, converged = gradient_descent_2d(
        g, gradient_g, initial_point=[-4.0, 4.0], learning_rate=learning_rate
    )
    results_2d[label] = (parameters, values, converged)
    print(
        f"{label:>9} (lr={learning_rate}): steps={len(parameters) - 1:4d}, "
        f"point={parameters[-1]}, g(point)={values[-1]:.6g}, converged={converged}"
    )

assert results_2d["moderate"][2]
np.testing.assert_allclose(results_2d["moderate"][0][-1], minimum_point, atol=1e-7)
assert not results_2d["too large"][2]

In [ ]:
x_values = np.linspace(-5, 6, 250)
y_values = np.linspace(-5, 6, 250)
X, Y = np.meshgrid(x_values, y_values)
Z = (X - 2) ** 2 + 2 * (Y + 1) ** 2

plt.figure(figsize=(8, 6))
plt.contour(X, Y, Z, levels=25)
for label, (parameters, _, _) in results_2d.items():
    plt.plot(parameters[:, 0], parameters[:, 1], marker=".", markersize=2, label=label)
plt.scatter(*minimum_point, color="black", marker="x", s=80, label="minimum")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Two-variable gradient descent paths")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
for label, (_, values, _) in results_2d.items():
    plt.plot(range(len(values)), np.maximum(values, 1e-16), label=label)
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("g(x, y) (log scale)")
plt.title("Two-variable objective values")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

For the moderate learning rate, the path approaches `(2, -1)` and the objective approaches zero. The too-large rate overshoots repeatedly along the steep direction and diverges. A large rate does not always fail for every function: the safe range depends on the function's curvature.

## Exercises

1. Change the initial point in each example. Does the converged solution change? Does the number of steps change?
2. Try a learning rate close to the stability limit for each function. Describe the path and convergence speed.
3. Change the stopping tolerance and explain how it affects the number of iterations.
4. Why is it not enough to report only the final parameter value? Name the configuration details needed to reproduce a run.

### Answer key (read after attempting)

For these strictly convex quadratics, any initial point converges to the same unique minimum when the learning rate is in the stable range. The path length and number of iterations can change. Rates near the stability boundary may converge slowly or oscillate. A complete report includes the function and gradient, initial point, learning rate, stopping rule, maximum iterations, and observed convergence.